1. **Load**

In [23]:
import re
import unicodedata
import pandas as pd
import spacy

from pathlib import Path
from spacy.language import Language
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span
from negspacy.negation import Negex
from negspacy.termsets import termset
from scispacy.linking import EntityLinker

fyp_dir = Path.cwd().parent
file_path = fyp_dir / "data"

train_df = pd.read_csv(f"{file_path}/train_split.csv")
test_df  = pd.read_csv(f"{file_path}/test_split.csv")

# train_df.head()

2. **Cleansing**

In [24]:
ABBR_MAP = {
    "cp": "chest pain",
    "c/p": "chest pain",

    "sob": "shortness of breath",
    "s.o.b": "shortness of breath",
    "soa": "shortness of breath",

    "n/v": "nausea vomiting",
    "n/vom": "nausea vomiting",
    "n/vomiting": "nausea vomiting",
    "n/v/d": "nausea vomiting diarrhea",

    "abd pain": "abdominal pain",
    "ap": "abdominal pain",

    "ha": "headache",
    "h/a": "headache",

    "diz": "dizziness",
    "loc": "loss of consciousness",
    "syn": "syncope",
    "faint": "syncope",

    "uti": "urinary tract infection",
    "uti.": "urinary tract infection",
    "uti/s": "urinary tract infection",
    "uti?": "urinary tract infection",

    "hematuria": "hematuria",
}

def normalize_abbr(text):
    if text is None or pd.isna(text):
        return ""

    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower().strip()

    # Normalize common punctuation variants.
    text = text.replace("’", "'")
    text = text.replace("(", " ")
    text = text.replace(")", " ")
    text = text.replace(",", " ")
    text = text.replace(";", " ")
    text = text.replace("|", " ")

    # Normalize repeated whitespace before abbreviation matching.
    text = re.sub(r"\s+", " ", text).strip()

    # Match longest abbreviations first.
    for abbreviation, expansion in sorted(
        ABBR_MAP.items(),
        key=lambda item: len(item[0]),
        reverse=True,
    ):
        pattern = rf"(?<!\w){re.escape(abbreviation)}(?!\w)"
        text = re.sub(pattern, expansion, text)

    # Remove unnecessary trailing punctuation.
    text = re.sub(r"[.!?]+$", "", text)

    # Collapse whitespace introduced by replacements.
    text = re.sub(r"\s+", " ", text).strip()

    return text

def clean_text(value):
    return normalize_abbr(value)


# # Test case
# ABBR_TESTS = {
#     "CP": "chest pain",
#     "SOB": "shortness of breath",
#     "c/p and n/v": "chest pain and nausea vomiting",
#     "abd pain, denies N/V": (
#         "abdominal pain denies nausea vomiting"
#     ),
#     "UTI?": "urinary tract infection",
# }

# for raw, expected in ABBR_TESTS.items():
#     actual = clean_text(raw)
#     print(f"{raw!r} -> {actual!r}")


------------------------------------------------------------------------

3. **Build a negation-aware spaCy pipeline**


a.) Create a spaCy component for Negex

In [25]:
from negspacy.negation import Negex
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span
from spacy.language import Language

SYMPTOM_KEYWORDS = [
    "chest pain",
    "shortness of breath",
    "dyspnea",
    "cough",
    "fever",
    "vomiting",
    "nausea",
    "abdominal pain",
    "back pain",
    "headache",
    "dizziness",
    "syncope",
    "weakness",
    "numbness",
    "stroke",
    "bleeding",
    "blood in stool",
    "hematemesis",
    "suicidal",
    "overdose",
    "burn",
    "urinary tract infection",
    "dysuria",
    "hematuria",
    "seizure",
    "neck pain",
    "palpitations",
    "diarrhea",
    "dehydration",
]

# Load spacy
nlp = spacy.load("en_core_web_sm", disable=["ner"],)

# Create matcher
matcher = PhraseMatcher(nlp.vocab, attr='LOWER')

patterns = [
    nlp.make_doc(keyword) for keyword in SYMPTOM_KEYWORDS
]

matcher.add("SYMPTOM", patterns)

# Custom Component
@Language.component("keyword_adder")
def keyword_adder(doc):
    matches = matcher(doc)

    spans = [Span(doc, start, end, label="SYMPTOM") for _, start, end in matches]

    # remove overlaps: keep longest spans (simple heuristic)
    spans = sorted(spans, key=lambda s: (s.start, - (s.end - s.start)))

    selected = []
    occupied = set()

    for s in spans:
        token_ids = set(range(s.start, s.end))

        if occupied.isdisjoint(token_ids):
            selected.append(s)
            occupied.update(token_ids)

    doc.ents = selected
    return doc

# Add pipeline 1
if "keyword_adder" in nlp.pipe_names:
    nlp.remove_pipe("keyword_adder")

nlp.add_pipe("keyword_adder", last=True)

# # test case
# text = "The patient has fever but denies chest pain."
# doc = nlp(text)

# for ent in doc.ents:
#     print(ent.text, ent._.negex)

c:\Users\ngngo\AppData\Local\Programs\Python\Python312\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


<function __main__.keyword_adder(doc)>

## Configure negation explicitly

In [26]:
"""
The order is important:
text
    -> keyword_adder creates entities
    -> negex assigns negation status
"""

# Pipeline 1
clinical_termset = termset("en_clinical")
negation_patterns = clinical_termset.get_patterns()

# Add local patterns useful for chief complaints.
negation_patterns["preceding_negations"].extend([
    "denies",
    "deny",
    "without",
    "negative for",
    "no",
    "not experiencing",
    "not reporting",
])

negation_patterns["following_negations"].extend([
    "denied",
    "absent",
])

negation_patterns["termination"].extend([
    "but",
    "however",
    "although",
    "except",
])


# Remove duplicates
for key in negation_patterns:
    negation_patterns[key] = list(set(negation_patterns[key]))

# Add negex for pipeline 1
if "negex" in nlp.pipe_names:
    nlp.remove_pipe("negex")

nlp.add_pipe(
    "negex",
    last=True,
    config={
        "ent_types": ["SYMPTOM"],
        "neg_termset": negation_patterns,
        "chunk_prefix": ["no"],
    },
)


# # Test case
# test_text = (
#     "patient denies chest pain and shortness of breath "
#     "no fever but reports dizziness"
# )

# doc = nlp(clean_text(test_text))

# for ent in doc.ents:
#     print(
#         ent.text,
#         "negated =", ent._.negex,
#         "start =", ent.start_char,
#         "end =", ent.end_char,
#     )


## scispaCy-style UMLS linking

In [ ]:
# Load scispacy
nlp_umls = spacy.load("en_core_sci_sm")

if "scispacy_linker" in nlp_umls.pipe_names:
    nlp_umls.remove_pipe("scispacy_linker")

# add UMLS linker pipeline 2
nlp_umls.add_pipe(
    "scispacy_linker",
    config={
        "linker_name": "umls",
        "resolve_abbreviations": True,
        "threshold": 0.80,
    }
)

# Add negex to pipeline 2
if "negex" in nlp_umls.pipe_names:
    nlp_umls.remove_pipe("negex")

nlp_umls.add_pipe(
    "negex",
    last=True,
    config={
        "ent_types": ["ENTITY"],   # or remove ent_types if needed
        "neg_termset": negation_patterns,
        "chunk_prefix": ["no"],
    },
)

linker = nlp_umls.get_pipe("scispacy_linker")


## Custom Symptom_List Pipline

In [30]:
# Extract Evidence (pipeline 1)
def extract_evidence(doc):
    """
    Returns list of dicts:
      {
        "concept": <keyword or matched phrase in lowercase>,
        "negated": bool,
        "span_text": <surface form>,
        "char_start": int,
        "char_end": int
      }
    """

    evidence = []
    for ent in doc.ents:
        negated = bool(ent._.negex)

        evidence.append({
            "concept": ent.text.lower().strip(),
            "negated": negated,
            "span_text": ent.text,
            "char_start": ent.start_char,
            "char_end": ent.end_char,
        })

    return evidence

# Define evidence extract function (pipeline 2)
def umls_link_evidence(doc, linker):
    """
    Returns list of UMLS-linked evidence dicts:
      {
        "concept": <canonical_name in lowercase>,
        "negated": bool,
        "span_text": <surface form>,
        "char_start": int,
        "char_end": int,
      }
    """
    
    evidence = []

    for ent in doc.ents:
        if not ent._.kb_ents:
            continue

        cui, score = ent._.kb_ents[0]
        kb_ent = linker.kb.cui_to_entity[cui]

        evidence.append({
            "concept": kb_ent.canonical_name.lower().strip(),
            "negated": bool(ent._.negex),
            "span_text": ent.text,
            "char_start": ent.start_char,
            "char_end": ent.end_char,
        })

    return evidence

## Combine two pipelines into one pipeline

In [31]:
# Preserve order if needed
def unique_preserve_order(values):
    seen = set()
    output = []

    for value in values:
        value = value.lower().strip()

        if value and value not in seen:
            output.append(value)
            seen.add(value)

    return output

# single-text preprocessing function
def preprocess_single(text: str, linker):
    normalized_text = clean_text(text)
    doc = nlp(normalized_text)
    doc_umls = nlp_umls(normalized_text)

    keyword_evidence = extract_evidence(doc)
    umls_evidence = umls_link_evidence(doc_umls, linker)

    evidence = keyword_evidence + umls_evidence

    affirmed = unique_preserve_order([
        item["concept"]
        for item in evidence
        if not item["negated"]
    ])

    negated = unique_preserve_order([
        item["concept"]
        for item in evidence
        if item["negated"]
    ])

    return {
        "text": normalized_text,
        "evidence": evidence,
        "affirmed_concepts": affirmed,
        "negated_concepts": negated,
    }

------------------------------------------------------------------------

## Testing

In [35]:
result = preprocess_single(
    "chest pain but denies shortness of breath"
, linker)
result

{'text': 'chest pain but denies shortness of breath',
 'evidence': [{'concept': 'chest pain',
   'negated': False,
   'span_text': 'chest pain',
   'char_start': 0,
   'char_end': 10},
  {'concept': 'shortness of breath',
   'negated': True,
   'span_text': 'shortness of breath',
   'char_start': 22,
   'char_end': 41},
  {'concept': 'chest pain',
   'negated': False,
   'span_text': 'chest pain',
   'char_start': 0,
   'char_end': 10},
  {'concept': 'short stature',
   'negated': True,
   'span_text': 'shortness',
   'char_start': 22,
   'char_end': 31}],
 'affirmed_concepts': ['chest pain'],
 'negated_concepts': ['shortness of breath', 'short stature']}

------------------------------------------------------------------------

## Conduct to Train and Test


In [33]:
# Define apply function
def preprocess_dataframe(df, source_col="chiefcomplaint"):
    out = df.copy()

    # Apply normalization
    texts = out[source_col].fillna("").map(clean_text).tolist()

    results = []

    for doc in nlp.pipe(texts, batch_size=256):
        evidence = extract_evidence(doc)

        affirmed = unique_preserve_order([
            item["concept"]
            for item in evidence
            if not item["negated"]
        ])

        negated = unique_preserve_order([
            item["concept"]
            for item in evidence
            if item["negated"]
        ])

        results.append({
            "text": doc.text,
            "evidence": evidence,
            "affirmed_concepts": affirmed,
            "negated_concepts": negated,
        })

    result_df = pd.DataFrame(results, index=out.index)

    out["text"] = result_df["text"]
    out["evidence"] = result_df["evidence"]
    out["affirmed_concepts"] = result_df["affirmed_concepts"]
    out["negated_concepts"] = result_df["negated_concepts"]

    return out

# Run it
train_processed = preprocess_dataframe(train_df)
test_processed = preprocess_dataframe(test_df)

In [34]:
train_processed.to_csv(f"{fyp_dir}/data/train_processed.csv", index=False)
test_processed.to_csv(f"{fyp_dir}/data/test_processed.csv", index=False)

print("Saved: train_processed.csv and test_processed.csv")

PermissionError: [Errno 13] Permission denied: 'c:\\Users\\ngngo\\OneDrive\\桌面\\Work\\Freelance\\Greech\\FYP/data/test_processed.csv'

------------------------------------------------------------------------

## Quality Checks

In [ ]:
def validate_row(row):
    affirmed = set(row["affirmed_concepts"])
    negated = set(row["negated_concepts"])

    assert affirmed.isdisjoint(negated), (
        "A concept appears as both affirmed and negated."
    )

    evidence_concepts = {
        item["concept"]
        for item in row["evidence"]
    }

    assert affirmed.issubset(evidence_concepts)
    assert negated.issubset(evidence_concepts)

    return True


for _, row in test_processed.head(1000).iterrows():
    validate_row(row)


from collections import Counter

affirmed_counts = Counter(
    concept
    for concepts in train_processed["affirmed_concepts"]
    for concept in concepts
)

negated_counts = Counter(
    concept
    for concepts in train_processed["negated_concepts"]
    for concept in concepts
)

print("Affirmed concepts:")
print(affirmed_counts.most_common(20))

print("\nNegated concepts:")
print(negated_counts.most_common(20))

unmatched = train_processed[
    train_processed["affirmed_concepts"].map(len).eq(0) &
    train_processed["negated_concepts"].map(len).eq(0)
]

unmatched[["chiefcomplaint", "text"]].head(50)

Affirmed concepts:
[('abdominal pain', 38649), ('chest pain', 25230), ('dyspnea', 16500), ('nausea', 15878), ('vomiting', 15251), ('back pain', 13259), ('headache', 12114), ('fever', 10883), ('weakness', 9841), ('dizziness', 7709), ('cough', 6993), ('diarrhea', 6782), ('syncope', 4734), ('neck pain', 3692), ('palpitations', 3642), ('numbness', 3211), ('seizure', 3202), ('bleeding', 2812), ('dysuria', 2715), ('shortness of breath', 2464)]

Negated concepts:
[('stroke', 48), ('urinary tract infection', 46), ('seizure', 8), ('dehydration', 3), ('abdominal pain', 3), ('chest pain', 2), ('nausea', 1), ('bleeding', 1), ('weakness', 1), ('diarrhea', 1)]


,chiefcomplaint,text
2,HYPOXIA,hypoxia
7,RLQ PAIN,rlq pain
8,FINGER LAC,finger lac
9,INCONTINENCE,incontinence
12,Agitation,agitation
13,"MVC, Rib pain",mvc rib pain
14,Pulseless foot,pulseless foot
15,SORE THROAT,sore throat
16,L THUMB LACERATION,l thumb laceration
18,FALL R WRIST PAIN,fall r wrist pain
